In [1]:
import random

from rdkit import Chem

from stereomolgraph import AtomId, Bond, MolGraph, StereoMolGraph
from stereomolgraph.experimental.canon import canon_atom_num
from stereomolgraph.ipython import View2D

View2D.show_h = True

In [2]:
def shuffle_graph(g: StereoMolGraph) -> StereoMolGraph:
    random_atom_ids: list[AtomId] = random.sample(range(len(g)), len(g))

    old_new_mapping: dict[AtomId | None, AtomId | None] = {None: None}
    new_old_mapping: dict[AtomId | None, AtomId | None] = {None: None}

    new_g = g.__class__()

    for atom, atom_type in random.sample(list(zip(g.atoms, g.atom_types)), len(g)):
        new_atom_id = random_atom_ids.pop()
        old_new_mapping[atom] = new_atom_id
        new_old_mapping[new_atom_id] = atom
        new_g.add_atom(new_atom_id, atom_type)

    for a1, a2 in random.sample(list(g.bonds), len(g.bonds)):
        new_g.add_bond(old_new_mapping[a1], old_new_mapping[a2])

    if isinstance(g, StereoMolGraph):
        for stereo in g.atom_stereo.values():
            new_stereo = stereo.__class__(
                tuple(old_new_mapping.get(atom) for atom in stereo.atoms), stereo.parity
            )

            new_g.set_atom_stereo(new_stereo)

        for stereo in g.bond_stereo.values():
            new_stereo = stereo.__class__(
                tuple(old_new_mapping.get(atom) for atom in stereo.atoms), stereo.parity
            )

            new_g.set_bond_stereo(new_stereo)

    return new_g


In [3]:
from stereomolgraph.stereodescriptors import OInt


def graph_to_set(
    g: StereoMolGraph,
) -> frozenset[tuple[AtomId, int] | Bond | tuple[str, int, tuple[OInt, ...]]]:
    ret: set[tuple[AtomId, int] | Bond | tuple[str, int, tuple[OInt, ...]]] = {
        (a, t) for a, t in zip(g.atoms, g.atom_types)
    }
    for bond in g.bonds:
        ret.add(bond)
    if isinstance(g, StereoMolGraph):
        for stereo in g.stereo.values():
            if None in stereo.atoms:
                # chiral lone pairs are described using "None"
                # To make it comparible it is substituted by len(graph)
                stereo = stereo.__class__(
                    tuple(a if a is not None else len(g) + 1 for a in stereo.atoms),
                    stereo.parity,
                )
            ret.add((stereo.__class__.__name__, *stereo.canonical_form()))
    return frozenset(ret)

In [4]:
def smiles2graph(smiles: str) -> StereoMolGraph:
    rdmol = Chem.AddHs(Chem.MolFromSmiles(smiles))
    smg = StereoMolGraph.from_rdmol(rdmol, stereo_complete=True)
    # smg = MolGraph.from_rdmol(rdmol)
    return smg

In [5]:
from pprint import pprint


def test_canon_atom_num(g: MolGraph, n: int = 10) -> bool:
    unique_graphs = set()
    for _ in range(n):
        shuffeled_g = shuffle_graph(g)
        canon_id_mapping = canon_atom_num(shuffeled_g)
        canon_graph = shuffeled_g.relabel_atoms(canon_id_mapping)
        unique_graphs.add(graph_to_set(canon_graph))
        # assert {len(ug) for ug in unique_graphs} == {len(g.atoms) + len(g.bonds)}
    if len(unique_graphs) > 1:
        tup = tuple(unique_graphs)
        display(g)
        pprint(tup[0].symmetric_difference(tup[1]))
    return True if len(unique_graphs) == 1 else False

In [6]:
ids_to_skip = [
    7,
    22,
    119,
    175,
    176,
    177,
    178,
    179,
    180,
    181,
    182,
    183,
    184,
    185,
    186,
    187,  # Isotopes
    10,
    11,  # Helicenes
    63,
    78,
    79,
    118,
    120,
    135,
    141,
    144,
    154,
    164,
    166,
    231,
    232,
    243,
    287,  # Allenes, Commulenes
    23,
    55,
    57,
    72,
    73,
    86,
    124,
    155,
    158,  # Atrop
]

In [7]:
# Read SMILES from text file
def load_smiles_from_txt(filepath: str) -> list[str]:
    """Load SMILES strings from a text file"""
    with open(filepath, "r") as f:
        smiles_list = [line.strip() for line in f if line.strip()]
    print(f"Loaded {len(smiles_list)} SMILES from {filepath}")
    return smiles_list


# Load SMILES from file
smiles_file = load_smiles_from_txt("./smiles.txt")
smiles_list = [
    smiles_file[i - 1].split()[0]
    for i, _ in enumerate(smiles_file, start=1)
    if i not in ids_to_skip
]

Loaded 300 SMILES from ./smiles.txt


Examples from: 
Krotko, D.G. Atomic ring invariant and Modified CANON extended connectivity algorithm for symmetry perception in molecular graphs and rigorous canonicalization of SMILES. J Cheminform 12, 48 (2020). https://doi.org/10.1186/s13321-020-00453-4

In [8]:
for smiles in smiles_list:
    try:
        g = smiles2graph(smiles)
    except Exception as e:
        print(f"Failed to parse SMILES: {smiles}")
        print(e)
        continue
    try:
        assert test_canon_atom_num(g), smiles
        print(f"Passed for SMILES: {smiles}")
    except AssertionError as e:
        print(f"Failed for SMILES: {smiles}")
        print(e)

Passed for SMILES: C1[C@]2(CCCC1)CC=CC2
Passed for SMILES: C1=CCC=CC=C1
Passed for SMILES: C[S@@](=O)C
Passed for SMILES: C1=CC=C2C(=C1)CCC=C2
Passed for SMILES: OC=1C=CC=CC1[C@H](C2=CC=CC=C2O)O
Passed for SMILES: C1[C@H]2C[C@H]3C[C@@H]1C[C@@H](C2)C3
Passed for SMILES: C=1[C@@H]2C=C[C@H](C1)C=C2
Passed for SMILES: [C@@H]12[C@H]3[C@H]4[C@@H]1[C@@H]5[C@H]4[C@H]3[C@H]25
Passed for SMILES: C1=C2CC=C3[C@]24C(CC=C4C1)=CC3
Passed for SMILES: CC\C(\C(\C)=N\O)=N\O
Failed to parse SMILES: [O-][N+](C=1C=CC(=CC1)[S@@](OCC)=O)=O

Passed for SMILES: [SiH3][C@]([GeH3])(OC)SC
Passed for SMILES: O=C([C@H]([C@H]([C@@H](C(O)=O)Cl)Cl)Cl)O
Passed for SMILES: O=C([C@@H]([C@H]([C@H](C(O)=O)Cl)Cl)Cl)O
Passed for SMILES: C1[C@@H]2C=C[C@H](CCCCCC\C=C\C1)CC2
Passed for SMILES: C1CCCCC\C=C\C1
Passed for SMILES: CCCCCCCCCC/C(=C(\C#N)/Br)/I
Passed for SMILES: C[C@H](CC)O
Passed for SMILES: C[C@H]1[C@@H]2CC[C@H]1[C@H](C2=O)Br
Passed for SMILES: FC(CN1[C@@H](C2=CC=CS2)CCC1)(F)F
Passed for SMILES: O[C@@H]/1CC/C=C\CC[C

KeyboardInterrupt: 